## Signal Check 1 — Previous Impressions

Signal:
Previous 30-day impressions (imp_prev30)

Reason:
Pages with higher historical impressions generally have more search visibility. A significant drop in recent impressions may indicate that the content requires refreshing.

Expected relationship:
Higher historical impressions combined with declining recent impressions should increase refresh priority.

In [ ]:
import pandas as pd

# Bucket previous impressions
data["imp_bucket"] = pd.qcut(
    data["imp_prev30"],
    q=5,
    duplicates="drop"
)

bucket1 = (
    data
    .groupby("imp_bucket")
    .agg(
        Pages=("imp_prev30","count"),
        Avg_Previous_Impressions=("imp_prev30","mean"),
        Decline_Rate=("is_declining","mean")
    )
)

print(bucket1)

print("\nTotal Pages:", len(data))

## Signal Check 2 — Average Search Position

Signal:
Average search position.

Reason:
Poor ranking positions often reduce visibility and may indicate content requiring improvement.

Expected relationship:
Pages with worse average positions should generally show higher refresh priority.

In [ ]:
data["position_bucket"] = pd.qcut(
    data["avg_position"],
    q=5,
    duplicates="drop"
)

bucket2 = (
    data
    .groupby("position_bucket")
    .agg(
        Pages=("avg_position","count"),
        Avg_Position=("avg_position","mean"),
        Avg_CTR=("ctr","mean")
    )
)

print(bucket2)

print("\nTotal Pages:", len(data))

## Baseline Refresh Rule

This baseline uses manually defined thresholds rather than machine learning.

The rule increases the priority score for pages with:

- High previous impressions
- Lower recent performance
- Poor average search position

The goal is to produce a transparent ranking that content teams can easily understand.

In [ ]:
# Baseline score

data["baseline_score"] = 0

data.loc[data["imp_prev30"] > 500, "baseline_score"] += 2

data.loc[data["avg_position"] > 10, "baseline_score"] += 1

data.loc[data["is_declining"] == 1, "baseline_score"] += 3

In [ ]:
def reason(row):

    if row["is_declining"] == 1:
        return "Traffic Declining"

    elif row["avg_position"] > 10:
        return "Poor Ranking"

    else:
        return "Monitor"

data["reason_code"] = data.apply(reason, axis=1)

In [ ]:
def action(row):

    if row["baseline_score"] >= 5:
        return "Refresh Immediately"

    elif row["baseline_score"] >= 3:
        return "Review Soon"

    else:
        return "Monitor"

data["action"] = data.apply(action, axis=1)

In [ ]:
queue = (
    data
    .sort_values(
        "baseline_score",
        ascending=False
    )
)

queue[
    [
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(20)

In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully.")

In [ ]:
top10 = queue.head(10)

top10[
    [
        "baseline_score",
        "reason_code",
        "action"
    ]
]

### Review of Top 10 Results

The highest-ranked pages were prioritized because they combine declining traffic with historically important search visibility.

Recommended Action:
Refresh Immediately

Possible reason the recommendation could be wrong:
Seasonality, temporary search fluctuations, or external events rather than outdated content.

These recommendations should therefore be treated as decision-support rather than automatic publishing decisions.

## Weaknesses of the Baseline

The rule uses manually selected thresholds that may not generalize across all clients.

Interactions between multiple variables are ignored.

The rule cannot learn from historical examples.

These limitations motivate the use of machine learning in the next phase.